### Determining the optimal number of hidden layers and neurons for an Artificial Neural Network (ANN) 
This can be challenging and often requires experimentation. However, there are some guidelines and methods that can help you in making an informed decision:

- Start Simple: Begin with a simple architecture and gradually increase complexity if needed.
- Grid Search/Random Search: Use grid search or random search to try different architectures.
- Cross-Validation: Use cross-validation to evaluate the performance of different architectures.
- Heuristics and Rules of Thumb: Some heuristics and empirical rules can provide starting points, such as:
  -    The number of neurons in the hidden layer should be between the size of the input layer and the size of the output layer.
  -  A common practice is to start with 1-2 hidden layers.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.pipeline import Pipeline
from scikeras.wrappers import KerasClassifier
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping
import pickle

In [2]:
data=pd.read_csv('Churn_Modelling.csv')
data = data.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)

label_encoder_gender = LabelEncoder()
data['Gender'] = label_encoder_gender.fit_transform(data['Gender'])

onehot_encoder_geo = OneHotEncoder(handle_unknown='ignore')
geo_encoded = onehot_encoder_geo.fit_transform(data[['Geography']]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded, columns=onehot_encoder_geo.get_feature_names_out(['Geography']))

data = pd.concat([data.drop('Geography', axis=1), geo_encoded_df], axis=1)

X = data.drop('Exited', axis=1)
y = data['Exited']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Save encoders and scaler for later use
with open('label_encoder_gender.pkl', 'wb') as file:
    pickle.dump(label_encoder_gender, file)

with open('onehot_encoder_geo.pkl', 'wb') as file:
    pickle.dump(onehot_encoder_geo, file)

with open('scaler.pkl', 'wb') as file:
    pickle.dump(scaler, file)


## Creating a Flexible ANN Model

### Purpose
- Define a function that creates an ANN model with customizable parameters.
- Used for **hyperparameter tuning** with `KerasClassifier`.
- Allows testing different numbers of neurons and hidden layers without rewriting the model.

### Function Parameters
- `neurons=32` → Number of neurons in each hidden layer.
- `layers=1` → Number of hidden layers in the network.

### Model Architecture
- Create a `Sequential` neural network.
- Add the first hidden layer with:
  - `neurons` neurons
  - `ReLU` activation
  - Input shape equal to the number of features.
- Use a loop to add additional hidden layers if `layers > 1`.
- Add an output layer with:
  - **1 neuron**
  - **Sigmoid** activation (used for binary classification).

### Compile the Model
- **Optimizer:** Adam
- **Loss Function:** Binary Crossentropy
- **Evaluation Metric:** Accuracy

### Why Use a Function?
- Makes the model reusable.
- Simplifies hyperparameter tuning.
- Easily experiment with different:
  - Number of neurons
  - Number of hidden layers
  - Other parameters (optimizer, learning rate, activation, etc.).

### KerasClassifier
- Wraps a Keras model so it behaves like a Scikit-learn estimator.
- Enables the use of:
  - `GridSearchCV`
  - `RandomizedSearchCV`
  - Cross-validation
- Helps find the best hyperparameters automatically.
```


In [3]:
## Define a function to create the model and try different parameters(KerasClassifier)

def create_model(neurons=32,layers=1):
    model=Sequential()
    model.add(Dense(neurons,activation='relu',input_shape=(X_train.shape[1],)))

    for _ in range(layers-1):
        model.add(Dense(neurons,activation='relu'))

    model.add(Dense(1,activation='sigmoid'))
    model.compile(optimizer='adam',loss="binary_crossentropy",metrics=['accuracy'])

    return model



In [7]:
# Create a KerasClassifier object.
# It wraps the Keras model so it can be used like a Scikit-learn estimator.

model = KerasClassifier(
    # Number of hidden layers to use in the ANN.
    layers=1,

    # Number of neurons in each hidden layer.
    neurons=32,

    # Function that builds and returns the Keras model.
    build_fn=create_model,

    # Display training progress during model fitting.
    # verbose=0 -> No output
    # verbose=1 -> Progress bar
    # verbose=2 -> One line per epoch
    verbose=1
)

In [8]:
# Define the hyperparameters to test during Grid Search.
# GridSearchCV will try every possible combination of these values.

param_grid = {

    # Test different numbers of neurons in each hidden layer.
    'neurons': [16, 32, 64, 128],

    # Test different numbers of hidden layers.
    'layers': [1, 2],

    # Train the model for different numbers of epochs.
    'epochs': [50, 100]
}

## Grid Search with Cross Validation

### Purpose
- Automatically find the best combination of hyperparameters.
- Trains and evaluates multiple ANN models using different parameter values.

### GridSearchCV Parameters

- **estimator=model**
  - The `KerasClassifier` model to optimize.

- **param_grid=param_grid**
  - Dictionary containing all hyperparameter combinations to test.

- **n_jobs=-1**
  - Uses all available CPU cores for parallel processing.
  - Speeds up the grid search.

- **cv=3**
  - Uses **3-Fold Cross Validation**.
  - Dataset is split into 3 parts.
  - Each model is trained 3 times, using a different validation fold each time.
  - Final score is the average of the 3 accuracies.

- **verbose=1**
  - Displays the progress of the grid search during execution.

### Training

```python
grid_result = grid.fit(X_train, y_train)
```

- Trains every hyperparameter combination.
- Performs cross-validation for each combination.
- Stores the results and identifies the best-performing model.

### Best Result

```python
print("Best: %f using %s" % (grid_result.best_score_, grid_result.best_params_))
```

- **best_score_**
  - Highest average cross-validation accuracy.

- **best_params_**
  - Hyperparameter combination that achieved the best score.

### Example Output

```text
Best: 0.865000 using {'epochs': 100, 'layers': 2, 'neurons': 64}
```

### Workflow

1. Create the model (`KerasClassifier`)
2. Define hyperparameter grid (`param_grid`)
3. Create `GridSearchCV`
4. Train using `grid.fit()`
5. Retrieve the best score and best parameters

### Note
If there are **16 hyperparameter combinations** and **cv = 3**, then:

- **Total model trainings = 16 × 3 = 48**
````


In [9]:
# Perform grid search
grid = GridSearchCV(estimator=model, param_grid=param_grid, n_jobs=-1, cv=3,verbose=1)
grid_result = grid.fit(X_train, y_train)

# Print the best parameters
print("Best: %f using %s" % (grid_result.best_score_, grid_result.best_params_))

Fitting 3 folds for each of 16 candidates, totalling 48 fits
Epoch 1/100


c:\Users\Omkar\Desktop\Python\.venv\Lib\site-packages\scikeras\wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
c:\Users\Omkar\Desktop\Python\.venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.7538 - loss: 0.5091
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8180 - loss: 0.4251
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8359 - loss: 0.4055
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8465 - loss: 0.3826
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8519 - loss: 0.3636
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8586 - loss: 0.3522
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8575 - loss: 0.3464
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8581 - loss: 0.3427
Epoch 9/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8602 - loss: 0.3399
Epoch 10/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8605 - loss: 0.3381
Epoch 11/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8605 - loss: 0.3366
Epoch 12/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step


## Grid Search Result

### Best Hyperparameters

- **Best Accuracy:** **0.858374 (85.84%)**
- **Best Epochs:** `100`
- **Best Hidden Layers:** `2`
- **Best Neurons:** `16` :contentReference[oaicite:0]{index=0}

### Interpretation

- The best-performing ANN uses:
  - **2 hidden layers**
  - **16 neurons** in each hidden layer
  - **100 training epochs**

- This combination achieved the highest **average cross-validation accuracy** among all tested hyperparameter combinations. :contentReference[oaicite:1]{index=1}

### Training Observation

- Training accuracy increased steadily during training.
- Loss decreased continuously.
- This indicates that the model was learning effectively. :contentReference[oaicite:2]{index=2} :contentReference[oaicite:3]{index=3}

### Conclusion

Use the following hyperparameters for the final ANN model:

- **Neurons:** `16`
- **Hidden Layers:** `2`
- **Epochs:** `100`
```
